In [1]:
!pip install requests beautifulsoup4 fpdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.0/327.0 kB 6.7 MB/s eta 0:00:008.4 MB/s eta 0:00:01


In [4]:
import requests
from bs4 import BeautifulSoup
from fpdf import FPDF
import time
import os
import re
from io import BytesIO
from PIL import Image

class TarotPDF(FPDF):
    def chapter_title(self, title):
        self.set_font('Helvetica', 'B', 18)
        self.cell(0, 10, title, 0, 1, 'L')
        self.ln(5)

    def chapter_body(self, body):
        self.set_font('Helvetica', '', 11)
        self.multi_cell(0, 6, body)
        self.ln()

    def add_infographic(self, img_url):
        try:
            response = requests.get(img_url)
            img = Image.open(BytesIO(response.content))
            # Convert to RGB if necessary to avoid PDF errors
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            # Temporary save to pass to FPDF
            img_path = "temp_tarot_img.jpg"
            img.save(img_path)
            
            # Scale image to fit page width (190mm)
            self.image(img_path, x=10, w=190)
            self.ln(10)
            os.remove(img_path)
        except Exception as e:
            print(f"Could not add image: {e}")

def clean_text(text):
    # Remove repetitive site-wide clutter and ads found in your previous PDF
    unwanted_patterns = [
        r"What is this\? Report Ad",
        r"If you are looking for a deck of Tarot cards.*here",
        r"For more information about.*description here",
        r"About the author:.*",
        r"Share\nShare\nLink\nClose share\nCopy link\nBack to blog"
    ]
    for pattern in unwanted_patterns:
        text = re.sub(pattern, "", text, flags=re.DOTALL | re.IGNORECASE)
    
    # Clean up excess whitespace
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

def scrape_tarot_reference():
    base_url = "https://playingcarddecks.com"
    index_url = "https://playingcarddecks.com/blogs/tarot-card-meanings?page="
    card_links = []

    print("Step 1: Gathering all card links...")
    for i in range(1, 14):
        res = requests.get(f"{index_url}{i}")
        soup = BeautifulSoup(res.text, 'html.parser')
        for link in soup.select('h4 a[href*="/blogs/tarot-card-meanings/"]'):
            url = base_url + link['href']
            if url not in card_links:
                card_links.append(url)
        time.sleep(0.5)

    pdf = TarotPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    print(f"Step 2: Extracting content for {len(card_links)} cards...")
    for url in card_links:
        print(f"Processing: {url.split('/')[-1]}")
        res = requests.get(url)
        soup = BeautifulSoup(res.text, 'html.parser')

        # 1. Extract Title
        title = soup.find('h1').get_text(strip=True).replace('Tarot Card Meanings with Infographic: ', '')
        
        # 2. Extract Featured Infographic Image
        img_tag = soup.select_one('.article-template__content img')
        img_url = img_tag['src'] if img_tag else None
        if img_url and img_url.startswith('//'):
            img_url = 'https:' + img_url

        # 3. Extract Main Article Body
        # We target the specific content div to avoid headers/footers
        content_div = soup.select_one('.article-template__content')
        if not content_div:
            continue

        # Remove video placeholders and sharing widgets before text extraction
        for junk in content_div.select('.social-sharing, .video-wrapper, script, style'):
            junk.decompose()

        raw_text = content_div.get_text(separator='\n')
        final_text = clean_text(raw_text)

        # 4. Add to PDF
        pdf.add_page()
        pdf.chapter_title(title)
        if img_url:
            pdf.add_infographic(img_url)
        pdf.chapter_body(final_text.encode('latin-1', 'ignore').decode('latin-1'))
        
        time.sleep(1) # Respectful crawl rate

    pdf.output("Tarot_Reference_Guide_2026.pdf")
    print("\nSuccess! Guide saved as Tarot_Reference_Guide_2026.pdf")

if __name__ == "__main__":
    scrape_tarot_reference()

Step 1: Gathering all card links...
Step 2: Extracting content for 78 cards...
Processing: tarot-card-meanings-with-infographic-the-devil
Processing: tarot-card-meanings-with-infographic-king-of-pentacles
Processing: tarot-card-meanings-with-infographic-queen-of-pentacles
Processing: tarot-card-meanings-with-infographic-knight-of-pentacles
Processing: tarot-card-meanings-with-infographic-page-of-pentacles
Processing: tarot-card-meanings-with-infographic-ten-of-pentacles
Processing: tarot-card-meanings-with-infographic-nine-of-pentacles
Processing: tarot-card-meanings-with-infographic-eight-of-pentacles
Processing: tarot-card-meanings-with-infographic-seven-of-pentacles
Processing: tarot-card-meanings-with-infographic-six-of-pentacles
Processing: tarot-card-meanings-with-infographic-five-of-pentacles
Processing: tarot-card-meanings-with-infographic-four-of-pentacles
Processing: tarot-card-meanings-with-infographic-three-of-pentacles
Processing: tarot-card-meanings-with-infographic-two-o

In [5]:
scrape_tarot_site()

Gathering card links from 13 pages...
Found 41 cards. Starting content extraction...
Processing: https://playingcarddecks.com/blogs/tarot-card-meanings/tarot-card-meanings-with-infographic-the-devil


/var/folders/8_/pxx4178x5qdb0psw7g51ygvh0000gn/T/ipykernel_81524/49514779.py:13: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 10, title, 0, 1, 'L')


Processing: https://playingcarddecks.com/blogs/tarot-card-meanings/tarot-card-meanings-with-infographic-king-of-pentacles
Processing: https://playingcarddecks.com/blogs/tarot-card-meanings/tarot-card-meanings-with-infographic-queen-of-pentacles
Processing: https://playingcarddecks.com/blogs/tarot-card-meanings/tarot-card-meanings-with-infographic-knight-of-pentacles
Processing: https://playingcarddecks.com/blogs/tarot-card-meanings/tarot-card-meanings-with-infographic-page-of-pentacles
Processing: https://playingcarddecks.com/blogs/tarot-card-meanings/tarot-card-meanings-with-infographic-ten-of-pentacles
Processing: https://playingcarddecks.com/blogs/tarot-card-meanings/tarot-card-meanings-with-infographic-nine-of-pentacles
Processing: https://playingcarddecks.com/blogs/tarot-card-meanings/tarot-card-meanings-with-infographic-eight-of-pentacles
Processing: https://playingcarddecks.com/blogs/tarot-card-meanings/tarot-card-meanings-with-infographic-seven-of-pentacles
Processing: https://